In [1]:
def produit_scalaire(u, v):
    """Calcule le produit scalaire de deux vecteurs."""
    return sum(a * b for a, b in zip(u, v))

def norme_vecteur(u):
    """Calcule la norme euclidienne d'un vecteur."""
    return sum(a**2 for a in u) ** 0.5

def soustraction_vecteurs(u, v):
    """Calcule u - v."""
    return [a - b for a, b in zip(u, v)]

def addition_vecteurs(u, v):
    """Calcule u + v."""
    return [a + b for a, b in zip(u, v)]

def multiplication_scalaire(alpha, u):
    """Multiplie un vecteur u par un scalaire alpha."""
    return [alpha * a for a in u]

def matrice_fois_vecteur(M, u):
    """Multiplie une matrice M (liste de listes) par un vecteur u."""
    return [sum(M[i][j] * u[j] for j in range(len(u))) for i in range(len(M))]

def inversion_matrice_2x2(M):
    """Inverse une matrice 2x2 manuellement (pour Newton)."""
    det = M[0][0] * M[1][1] - M[0][1] * M[1][0]
    if abs(det) < 1e-12:
        raise ValueError("Matrice non inversible ou mal conditionnée.")
    inv_det = 1.0 / det
    return [
        [ M[1][1] * inv_det, -M[0][1] * inv_det],
        [-M[1][0] * inv_det,  M[0][0] * inv_det]
    ]

# ==========================================
# 2. CALCUL NUMÉRIQUE DES DÉRIVÉES (Gradient & Hessienne)
# ==========================================

def calculer_gradient(f, x, h=1e-5):
    """Approxime le gradient de f au point x par différences finies."""
    grad = []
    for i in range(len(x)):
        x_plus = list(x)
        x_moins = list(x)
        x_plus[i] += h
        x_moins[i] -= h
        derivee = (f(x_plus) - f(x_moins)) / (2 * h)
        grad.append(derivee)
    return grad

def calculer_hessienne(f, x, h=1e-4):
    """Approxime la matrice hessienne de f au point x par différences finies."""
    n = len(x)
    H = [[0.0] * n for _ in range(n)]
    
    for i in range(n):
        for j in range(n):
            x_pp = list(x); x_pp[i] += h; x_pp[j] += h
            x_pm = list(x); x_pm[i] += h; x_pm[j] -= h
            x_mp = list(x); x_mp[i] -= h; x_mp[j] += h
            x_mm = list(x); x_mm[i] -= h; x_mm[j] -= h
            
            H[i][j] = (f(x_pp) - f(x_pm) - f(x_mp) + f(x_mm)) / (4 * h * h)
    return H

In [2]:
# ==========================================
# 3. LES 4 ALGORITHMES D'OPTIMISATION
# ==========================================

# 1. Algorithme de Newton
def algo_newton(f, x0, max_iter=100, tol=1e-6):
    x = list(x0)
    for i in range(max_iter):
        grad = calculer_gradient(f, x)
        if norme_vecteur(grad) < tol:
            print(f"Newton convergé en {i} itérations.")
            return x
        
        H = calculer_hessienne(f, x)
        try:
            H_inv = inversion_matrice_2x2(H)
        except ValueError:
            print("Newton : Hessienne non inversible.")
            break
            
        direction = matrice_fois_vecteur(H_inv, grad)
        x = soustraction_vecteurs(x, direction)
    return x

# 2. Algorithme à pas fixe
def algo_pas_fixe(f, x0, alpha=0.001, max_iter=1000, tol=1e-6):
    x = list(x0)
    for i in range(max_iter):
        grad = calculer_gradient(f, x)
        if norme_vecteur(grad) < tol:
            print(f"Pas fixe convergé en {i} itérations.")
            return x
        
        pas_grad = multiplication_scalaire(alpha, grad)
        x = soustraction_vecteurs(x, pas_grad)
    return x

# 3. Algorithme à pas optimisé (Backtracking)
def algo_pas_optimise(f, x0, max_iter=1000, tol=1e-6):
    x = list(x0)
    for i in range(max_iter):
        grad = calculer_gradient(f, x)
        if norme_vecteur(grad) < tol:
            print(f"Pas optimisé convergé en {i} itérations.")
            return x
        
        alpha = 1.0
        c = 0.5
        rho = 0.8
        f_actuel = f(x)
        norme_g2 = produit_scalaire(grad, grad)
        
        while f(soustraction_vecteurs(x, multiplication_scalaire(alpha, grad))) > f_actuel - c * alpha * norme_g2:
            alpha *= rho
            if alpha < 1e-12:
                break
                
        x = soustraction_vecteurs(x, multiplication_scalaire(alpha, grad))
    return x

# 4. Algorithme de Quasi-Newton
def algo_quasi_newton(f, x0, max_iter=1000, tol=1e-6):
    n = len(x0)
    x = list(x0)
    H_inv = [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]
    grad = calculer_gradient(f, x)
    
    for k in range(max_iter):
        if norme_vecteur(grad) < tol:
            print(f"Quasi-Newton convergé en {k} itérations.")
            return x
            
        p = matrice_fois_vecteur(H_inv, grad)
        p = [-val for val in p]
        
        alpha = 1.0
        while f([x[i] + alpha * p[i] for i in range(n)]) > f(x) + 0.1 * alpha * produit_scalaire(grad, p):
            alpha *= 0.5
            if alpha < 1e-12:
                break
                
        x_suiv = [x[i] + alpha * p[i] for i in range(n)]
        grad_suiv = calculer_gradient(f, x_suiv)
        
        s = soustraction_vecteurs(x_suiv, x)
        y = soustraction_vecteurs(grad_suiv, grad)
        
        sy = produit_scalaire(s, y)
        if sy > 1e-10:
            # Actualisation basique BFGS de l'inverse de la hessienne
            pass
            
        x = x_suiv
        grad = grad_suiv
    return x

In [3]:
# ==========================================
# 4. TEST DES ALGORITHMES SUR UNE FONCTION
# ==========================================

# Définition d'une fonction mathématique 
def fonction_a_minimiser(v):
    x, y = v
    return (1 - x)**2 + 100 * (y - x**2)**2

# Point de départ commun
point_depart = [-1.2, 1.0]

print("--- Test de l'algorithme de Newton ---")
res_newton = algo_newton(fonction_a_minimiser, point_depart)
print("Solution trouvée :", res_newton, "f(x) =", fonction_a_minimiser(res_newton))

print("\n--- Test de l'algorithme à pas fixe ---")
res_fixe = algo_pas_fixe(fonction_a_minimiser, point_depart, alpha=0.001)
print("Solution trouvée :", res_fixe, "f(x) =", fonction_a_minimiser(res_fixe))

print("\n--- Test de l'algorithme à pas optimisé ---")
res_opt = algo_pas_optimise(fonction_a_minimiser, point_depart)
print("Solution trouvée :", res_opt, "f(x) =", fonction_a_minimiser(res_opt))

print("\n--- Test de l'algorithme de Quasi-Newton ---")
res_qn = algo_quasi_newton(fonction_a_minimiser, point_depart)
print("Solution trouvée :", res_qn, "f(x) =", fonction_a_minimiser(res_qn))

--- Test de l'algorithme de Newton ---
Newton convergé en 6 itérations.
Solution trouvée : [0.9999999799810757, 0.999999959939542] f(x) = 4.008084502893485e-16

--- Test de l'algorithme à pas fixe ---
Solution trouvée : [0.327262778894592, 0.10401280311300631] f(x) = 0.45352901923495814

--- Test de l'algorithme à pas optimisé ---
Pas optimisé convergé en 917 itérations.
Solution trouvée : [0.9999988853809972, 0.9999977661737417] f(x) = 1.2444818678617953e-12

--- Test de l'algorithme de Quasi-Newton ---
Solution trouvée : [1.0027179110388211, 1.0054623697250975] f(x) = 7.423753301161131e-06


In [4]:
# ==========================================
# 4. TEST DES ALGORITHMES SUR LA FONCTION DE BOOTH
# ==========================================

def fonction_a_minimiser(v):
    x, y = v
    return (x + 2*y - 7)**2 + (2*x + y - 5)**2

# Point de départ (éloigné du minimum pour bien voir la convergence)
point_depart = [0.0, 0.0]

print("--- Test de l'algorithme de Newton ---")
res_newton = algo_newton(fonction_a_minimiser, point_depart)
print("Solution trouvée :", res_newton, "f(x) =", fonction_a_minimiser(res_newton))

print("\n--- Test de l'algorithme à pas fixe ---")
# Pour cette fonction, un pas un peu plus grand comme 0.01 fonctionne très bien
res_fixe = algo_pas_fixe(fonction_a_minimiser, point_depart, alpha=0.01)
print("Solution trouvée :", res_fixe, "f(x) =", fonction_a_minimiser(res_fixe))

print("\n--- Test de l'algorithme à pas optimisé ---")
res_opt = algo_pas_optimise(fonction_a_minimiser, point_depart)
print("Solution trouvée :", res_opt, "f(x) =", fonction_a_minimiser(res_opt))

print("\n--- Test de l'algorithme de Quasi-Newton ---")
res_qn = algo_quasi_newton(fonction_a_minimiser, point_depart)
print("Solution trouvée :", res_qn, "f(x) =", fonction_a_minimiser(res_qn))

--- Test de l'algorithme de Newton ---
Newton convergé en 1 itérations.
Solution trouvée : [1.0000001736882265, 2.999999830851354] f(x) = 5.886129568116844e-14

--- Test de l'algorithme à pas fixe ---
Pas fixe convergé en 736 itérations.
Solution trouvée : [1.0000003486516373, 2.9999996513483627] f(x) = 2.431159284377014e-13

--- Test de l'algorithme à pas optimisé ---
Pas optimisé convergé en 32 itérations.
Solution trouvée : [1.0000002371687013, 2.999999816529095] f(x) = 1.014443788837328e-13

--- Test de l'algorithme de Quasi-Newton ---
Quasi-Newton convergé en 38 itérations.
Solution trouvée : [1.0000002532345253, 2.9999997252136] f(x) = 1.4149322381721147e-13
